<a href="https://colab.research.google.com/github/MuhammadQasimTanveer/AL-ScientificCalculator/blob/main/symptoms_disease_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/MuhammadQasimTanveer/HerdOrbit-ML/blob/main/symptoms_disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Symptoms Disease Prediction using Random Forest

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print("Basic Setup completed")

Add dataset loading and basic setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load dataset
df = pd.read_excel('symptoms.xlsx')
print(df.head())

## Describe Dataset

In [ ]:
symptom_cols = ['Symptom 1', 'Symptom 2', 'Symptom 3']
symptoms = sorted(s for s in pd.unique(df[symptom_cols].values.ravel()) if s != 'none')

print('Total records  :', len(df))
print('Total columns  :', df.shape[1])
print('Total diseases :', df['Disease'].nunique())
print('Total animals  :', df['Animal'].nunique())
print('Total symptoms :', len(symptoms), '(plus "none" for healthy animals)')
print('Missing values :', df.isnull().sum().sum())

In [ ]:
df.describe()

Add data preprocessing and symptom combination

In [ ]:
df['All_Symptoms'] = df['Symptom 1'].astype(str) + " " + df['Symptom 2'].astype(str) + " " + df['Symptom 3'].astype(str)
df = df.drop(['Symptom 1', 'Symptom 2', 'Symptom 3'], axis=1)
print(df.head())

Add label encoding for categorical features

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_animal = LabelEncoder()
le_symptoms = LabelEncoder()
le_disease = LabelEncoder()

df['Animal'] = le_animal.fit_transform(df['Animal'].astype(str))
df['All_Symptoms'] = le_symptoms.fit_transform(df['All_Symptoms'].astype(str))
df['Disease'] = le_disease.fit_transform(df['Disease'].astype(str))

print("All columns encoded!")
print(df.dtypes)

Add train-test split and feature separation

In [ ]:
X = df.drop('Disease', axis=1)
y = df['Disease'].values   # already encoded in the previous cell

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print('Train :', X_train.shape, '-', X_train.shape[0], 'records,', X_train.shape[1], 'features')
print('Test  :', X_test.shape, '-', X_test.shape[0], 'records,', X_test.shape[1], 'features')
print('Total :', len(X_train) + len(X_test), 'records')

In [ ]:
print("Data types of X:")
print(X.dtypes)

## Feature Selection

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

# Mutual information of each feature with the disease
mi = pd.Series(mutual_info_classif(X_train, y_train, discrete_features=(X_train.columns != 'Temperature'),
                                   random_state=42), index=X_train.columns).sort_values()

plt.figure(figsize=(8, 4))
mi.plot(kind='barh', color='seagreen')
plt.title('Mutual information of each feature')
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(X_train.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Correlation heatmap of features')
plt.show()

## Model Training

In [ ]:
# Model training
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Model trained successfully!")

## Training Curves

In [ ]:
from sklearn.metrics import log_loss

# Loss and accuracy for different numbers of trees
x_values = [1, 2, 5, 10, 20, 30, 50, 75, 100]
train_acc, test_acc, train_loss, test_loss = [], [], [], []

m = RandomForestClassifier(n_estimators=1, warm_start=True, random_state=42, n_jobs=-1)
for n in x_values:
    m.set_params(n_estimators=n)
    m.fit(X_train, y_train)
    train_acc.append(m.score(X_train, y_train))
    test_acc.append(m.score(X_test, y_test))
    train_loss.append(log_loss(y_train, m.predict_proba(X_train), labels=m.classes_))
    test_loss.append(log_loss(y_test, m.predict_proba(X_test), labels=m.classes_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_values, train_loss, label='Train')
axes[0].plot(x_values, test_loss, label='Test', linestyle='--')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Number of trees')
axes[0].set_ylabel('Log loss')
axes[0].legend()

axes[1].plot(x_values, train_acc, label='Train')
axes[1].plot(x_values, test_acc, label='Test', linestyle='--')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Number of trees')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()


Add model evaluation and accuracy report

In [ ]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = {}
for name, Xp, yp in [('Train', X_train, y_train), ('Test', X_test, y_test)]:
    pred = model.predict(Xp)
    results[name] = {'Accuracy': accuracy_score(yp, pred),
                     'Precision': precision_score(yp, pred, average='macro'),
                     'Recall': recall_score(yp, pred, average='macro'),
                     'F1': f1_score(yp, pred, average='macro')}
pd.DataFrame(results).T.round(4)

 Save Trained Model and Encoders

In [ ]:
import joblib

joblib.dump(model, 'random_forest_model.pkl', compress=3)
joblib.dump(le_animal, 'le_animal.pkl')
joblib.dump(le_symptoms, 'le_symptoms.pkl')
joblib.dump(le_disease, 'le_disease.pkl')

print("Model and encoders saved successfully.")

Add Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_disease.classes_,
            yticklabels=le_disease.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Symptoms Disease Prediction')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_test, y_pred, normalize='true') * 100, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=le_disease.classes_, yticklabels=le_disease.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (%) - Symptoms Disease Prediction')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## ROC Curve

In [ ]:
from sklearn.metrics import auc, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

class_names = list(le_disease.classes_)
y_proba = model.predict_proba(X_test)
y_bin = label_binarize(y_test, classes=range(len(class_names)))

plt.figure(figsize=(9, 7))
for i, name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve (one-vs-rest) - Test')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.legend(loc='lower right')
plt.show()

print('Macro AUC - Test:', round(roc_auc_score(y_test, y_proba, multi_class='ovr'), 4))

 Add Sample Prediction Test

In [ ]:
sample = {
    'Animal': le_animal.transform(['cow'])[0],
    'Age': 3,
    'Temperature': 103.5,
    'All_Symptoms': le_symptoms.transform(['coughing nasal discharge loss of appetite'])[0]
}

sample_df = pd.DataFrame([sample])
prediction = model.predict(sample_df)
predicted_disease = le_disease.inverse_transform(prediction)

print("Predicted Disease:", predicted_disease[0])

Add Model Summary Markdown

In [ ]:
print('Records :', len(df), '| Diseases:', len(class_names), '| Features:', X.shape[1])
print('Split   : Train', len(X_train), '/ Test', len(X_test))
print('Test accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')